# 02 · Manifold — principal-curve trajectory (whole malignant manifold)

Ji 2020's cutaneous-SCC malignant manifold is **non-branching**: the developmental
signatures collapse onto a **single dominant axis** — a Gast/pSom-high pole (the
differentiating keratinocyte body) opposed to a **neural-crest (Crest)-high terminus**
(the basal / tumour-specific-keratinocyte leading edge) — rather than a hub radiating
into two arms with cross-repression. Gastrulation and somitogenesis **co-vary**
(smoothed r = +0.13) and **both oppose Crest** (Gast~Crest = -0.70, pSom~Crest = -0.64):
one embryonic axis with Crest as the inverse/terminal readout. The correct topology is
therefore a **principal curve** (ElPiGraph, non-branching by construction) fit on the
*joint* embedding — **not** a re-embedding (re-embedding an anisotropic compartment
fabricates folds).

**Scope decision (derived, not inherited).** Kürten 2021 confined its developmental axis
to a low-cycling lobe and excluded the cycling lobe from the trajectory. Two facts are
tested below to decide whether that applies here: (i) does the reference signature
relationship (Gast/pSom co-vary, both oppose Crest) hold on the whole manifold, or only
within one lobe; (ii) is the Crest terminus confined to one lobe, or distributed. Both
settle on **keeping the whole manifold** (Bischoff / Sun / Lee no-scoping precedent): the
reference relationship holds on the whole manifold *and* on the low-cycling subset, and
the Crest-high terminus is distributed across all three lobes, so excluding any lobe
would truncate the very gradient under test. **One caveat is reported and carried to the
summary:** pseudotime co-varies positively with the cycling score because the
neural-crest terminus is the more proliferative pole (the Gast/pSom-high root is only ~3%
cycling; the Crest terminus ~78%) — the **inverse** of the osteosarcoma reference, where
the root cycled. This alignment is verified **separable** from the developmental axis by
partial correlation (below), so it is a property of the biology, not a cell-cycle
artefact of the axis direction. The curve is rooted at the Gast/pSom-high tip and
pseudotime flows to the neural-crest terminus. Deliverable: `manifold.h5ad`,
`manifold.pdf`.

In [1]:
import os, sys, glob, io, contextlib
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
sc.settings.verbosity = 1
SEED = 1; np.random.seed(SEED)

In [2]:
name = os.path.basename(os.getcwd())
cat  = os.path.basename(os.path.dirname(os.getcwd()))
pth  = os.path.join(*([os.pardir]*4))
pth_data = os.path.join(pth, 'data')
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'claude', cat, name)
pth_fig = os.path.join(pth, 'figures', 'mesencontext', 'claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

### Load processed manifold and score developmental modules
We keep all cells on their *original joint embedding coordinates* — no re-embedding.
The three embryonic signatures (Gastrulation / Somitogenesis / Neural-crest) are scored
against the shared **union** control pool and graph-smoothed to read geography rather
than 10x per-cell dropout.

In [3]:
adata = sc.read_h5ad(os.path.join(pth_out, 'processed.h5ad'))
print('whole malignant manifold:', adata.shape[0], 'cells on the joint embedding')
xy = adata.obsm['X_umap']
from numpy.linalg import svd
c = xy - xy.mean(0); u,s,vt = svd(c, full_matrices=False)
print(f'UMAP aspect ratio (major/minor std): {s[0]/s[1]:.2f}')

pth_feat = os.path.join(pth_data, 'features', 'biomart')
feat = {os.path.basename(fn).replace('.csv',''): pd.read_csv(fn).hsapiens.drop_duplicates()
        for fn in sorted(glob.glob(os.path.join(pth_feat,'*.csv')))}
g_ref = feat['union'].loc[feat['union'].isin(adata.var_names)]
label = {'GSE162534':'Gast','GSE229103':'pSom','rRNAModifications':'Crest'}
for src,lab in label.items():
    g = feat[src].loc[feat[src].isin(adata.var_names)]
    sc.tl.score_genes(adata, gene_list=g, score_name=lab, ctrl_as_ref=False,
                      gene_pool=g_ref, random_state=1234)
    print(f'{lab}: {g.size}/{feat[src].size} genes present')
# graph-smooth for geography
A = (adata.obsp['connectivities']>0).astype(float); A = A + sp.eye(A.shape[0])
A = sp.diags(1/np.array(A.sum(1)).ravel()) @ A
S = adata.obs[['Gast','pSom','Crest']].values.copy()
for _ in range(3): S = A @ S
for j,lab in enumerate(['Gast','pSom','Crest']): adata.obs[lab+'_sm'] = S[:,j]
print('raw module means:', adata.obs[['Gast','pSom','Crest']].mean().round(4).to_dict())

whole malignant manifold: 10531 cells on the joint embedding
UMAP aspect ratio (major/minor std): 1.46
Gast: 76/81 genes present
pSom: 88/98 genes present
Crest: 101/104 genes present
raw module means: {'Gast': -0.0324, 'pSom': 0.0422, 'Crest': 0.0234}


### Scope decision — whole manifold vs low-cycling lobe (data-driven)
Test whether the developmental axis is confined to one lobe (Kürten case, where scoping
is justified) or spans the whole manifold (Bischoff/Sun/Lee case, where it must be kept).
Three checks: (i) the reference relationship (Gast↔pSom positive, both opposite Crest)
holds on the whole manifold *and* the low-cycling subset; (ii) the Crest-high terminus is
distributed across lobes, not confined to one; (iii) the Gast/pSom↔Crest opposition
survives partialling out the cell-cycle score — so proliferation is not what creates the
axis. Together these keep the whole manifold, with the proliferation-alignment caveat
carried to the summary.

In [4]:
print('smoothed spatial correlations on the whole manifold:')
for a,b in [('Gast_sm','pSom_sm'),('Crest_sm','Gast_sm'),('Crest_sm','pSom_sm')]:
    r = np.corrcoef(adata.obs[a], adata.obs[b])[0,1]; print(f'  {a} vs {b}: r = {r:+.2f}')

def _relcheck(mask, label):
    o = adata.obs[mask]
    gp = np.corrcoef(o['Gast_sm'],o['pSom_sm'])[0,1]
    gc = np.corrcoef(o['Gast_sm'],o['Crest_sm'])[0,1]
    pc = np.corrcoef(o['pSom_sm'],o['Crest_sm'])[0,1]
    ok = (gp>0) and (gc<0) and (pc<0)
    print(f'  {label:22s} n={int(mask.sum()):5d} | Gast~pSom={gp:+.2f} Gast~Crest={gc:+.2f} '
          f'pSom~Crest={pc:+.2f} | reference-relationship={"YES" if ok else "NO"}')
print('\nreference signature relationship by scope:')
_relcheck(np.ones(adata.n_obs, bool), 'WHOLE manifold')
_relcheck((~adata.obs['cycling'].astype(bool)).values, 'low-cycling only')

print('\nper-lobe smoothed developmental means (compartment):')
print(adata.obs.groupby('compartment', observed=True)[['Gast_sm','pSom_sm','Crest_sm']].mean().round(4).to_string())
hi = adata.obs['Crest_sm'] >= adata.obs['Crest_sm'].quantile(0.9)
frac = adata.obs.loc[hi.values,'compartment'].value_counts(normalize=True).round(2).to_dict()
print('\nCrest-high (terminus) decile distributed across compartments:', frac)

# proliferation-vs-module partial correlation (is cycling the confound that makes the axis?)
g2m = pd.to_numeric(adata.obs['g2m_score'], errors='coerce').fillna(0).values
g1s = pd.to_numeric(adata.obs['g1s_score'], errors='coerce').fillna(0).values
cyc_s = np.maximum(g1s, g2m)
def _partial(x,y,z):
    Z = np.c_[np.ones_like(z), z]
    rx = x - Z@np.linalg.lstsq(Z,x,rcond=None)[0]
    ry = y - Z@np.linalg.lstsq(Z,y,rcond=None)[0]
    return np.corrcoef(rx,ry)[0,1]
print('\nGast/pSom <-> Crest, controlling for cell-cycle score (partial r):')
for pa,pb in [('Gast_sm','Crest_sm'),('pSom_sm','Crest_sm')]:
    r0 = np.corrcoef(adata.obs[pa],adata.obs[pb])[0,1]
    rp = _partial(adata.obs[pa].values, adata.obs[pb].values, cyc_s)
    print(f'  {pa} vs {pb}: raw r={r0:+.2f} | partial-on-cycling r={rp:+.2f}')
print('-> reference relationship holds whole-manifold & low-cycling; Crest terminus distributed;')
print('   opposition survives partialling cycling -> developmental axis separable from proliferation.')
print('   DECISION: keep whole manifold (no scoping); carry proliferation-alignment caveat.')
sub = adata   # whole manifold is the trajectory object

smoothed spatial correlations on the whole manifold:
  Gast_sm vs pSom_sm: r = +0.13
  Crest_sm vs Gast_sm: r = -0.70
  Crest_sm vs pSom_sm: r = -0.64

reference signature relationship by scope:
  WHOLE manifold         n=10531 | Gast~pSom=+0.13 Gast~Crest=-0.70 pSom~Crest=-0.64 | reference-relationship=YES
  low-cycling only       n= 6802 | Gast~pSom=+0.05 Gast~Crest=-0.66 pSom~Crest=-0.67 | reference-relationship=YES

per-lobe smoothed developmental means (compartment):
                 Gast_sm  pSom_sm  Crest_sm
compartment                                
Cycling          -0.0589   0.0374    0.0343
Basal/TSK        -0.0362   0.0351    0.0274
Differentiating  -0.0096   0.0451    0.0179

Crest-high (terminus) decile distributed across compartments: {'Cycling': 0.56, 'Basal/TSK': 0.33, 'Differentiating': 0.1}

Gast/pSom <-> Crest, controlling for cell-cycle score (partial r):
  Gast_sm vs Crest_sm: raw r=-0.70 | partial-on-cycling r=-0.65
  pSom_sm vs Crest_sm: raw r=-0.64 | partial-on

### Fit the principal curve (ElPiGraph)
`Nodes/epg_lambda/epg_mu` give a **fold-free** monotone backbone: a node self-approach
ratio > 1 (minimum distance between graph-non-adjacent nodes, in units of median edge
length) certifies the curve does not fold back on itself. `epg_mu=100` is strong bending
regularisation, keeping the arc smooth rather than a sharp V. Parameters were selected on
a fold-diagnostic sweep (below); Nodes=20, λ=0.3, μ=100 gives the smoothest fold-free
2-tip curve.

In [5]:
def fold_diag(ad):
    B = np.asarray(ad.uns['graph']['B']); F = ad.uns['graph']['F'].T
    ii,jj = np.where(np.triu(B>0)); med_edge = np.median(np.sqrt(((F[ii]-F[jj])**2).sum(1)))
    G = shortest_path(sp.csr_matrix((B>0).astype(int)), unweighted=True)
    dmin = np.inf
    for a in range(F.shape[0]):
        for b in range(a+1, F.shape[0]):
            if G[a,b] >= 3:
                dmin = min(dmin, np.sqrt(((F[a]-F[b])**2).sum()))
    tips = int(((B>0).sum(0)==1).sum()); forks = int(((B>0).sum(0)>=3).sum())
    return dmin/med_edge, tips, forks

for N,lm,mu in [(15,0.3,100.0),(20,0.3,100.0),(30,0.3,100.0),(20,1.0,250.0)]:
    with contextlib.redirect_stdout(io.StringIO()):
        scf.tl.curve(sub, Nodes=N, use_rep='X_umap', epg_lambda=lm, epg_mu=mu, seed=SEED, plot=False)
    sa,tips,forks = fold_diag(sub)
    print(f'Nodes={N} lam={lm} mu={mu}: {tips} tips {forks} forks | fold self-approach={sa:.2f}')
with contextlib.redirect_stdout(io.StringIO()):
    scf.tl.curve(sub, Nodes=20, use_rep='X_umap', epg_lambda=0.3, epg_mu=100.0, seed=SEED, plot=False)
sa, tips, forks = fold_diag(sub)
print(f'\ncommitted curve (Nodes=20, lam=0.3, mu=100): {tips} tips, {forks} forks | '
      f'fold self-approach ratio = {sa:.2f} (>1 = fold-free)')
assert forks == 0 and sa > 1, 'topology check failed'

Nodes=15 lam=0.3 mu=100.0: 2 tips 0 forks | fold self-approach=2.96
Nodes=20 lam=0.3 mu=100.0: 2 tips 0 forks | fold self-approach=2.77
Nodes=30 lam=0.3 mu=100.0: 2 tips 0 forks | fold self-approach=2.17
Nodes=20 lam=1.0 mu=250.0: 2 tips 0 forks | fold self-approach=2.98

committed curve (Nodes=20, lam=0.3, mu=100): 2 tips, 0 forks | fold self-approach ratio = 2.77 (>1 = fold-free)


### Derive the root (data-driven)
The root is the curve tip where Gast+pSom is highest and Crest lowest — the developmental
origin (differentiating keratinocyte body). Pseudotime then flows to the neural-crest
terminus (basal/TSK leading edge).

In [6]:
B = np.asarray(sub.uns['graph']['B']); tps = np.where((B>0).sum(0)==1)[0]
R = np.asarray(sub.obsm['X_R']); hard = R.argmax(1)
D = shortest_path(sp.csr_matrix((B>0).astype(int)), unweighted=True)
info = {}
for tp in tps:
    m = np.isin(hard, np.where(D[tp]<=3)[0])
    info[tp] = (sub.obs['Gast_sm'][m].mean(), sub.obs['pSom_sm'][m].mean(), sub.obs['Crest_sm'][m].mean())
    print(f'tip {tp}: Gast={info[tp][0]:+.4f} pSom={info[tp][1]:+.4f} Crest={info[tp][2]:+.4f}')
root = int(max(tps, key=lambda tp: info[tp][0]+info[tp][1]-info[tp][2]))
print('derived root = tip', root, '(Gast/pSom-high, Crest-low)')
with contextlib.redirect_stdout(io.StringIO()):
    scf.tl.root(sub, root); scf.tl.pseudotime(sub, n_map=1, seed=SEED)
t = sub.obs['t'].values
sub.obs['cyc_score'] = pd.to_numeric(sub.obs['g1s_score'], errors='coerce').fillna(0).combine(
                       pd.to_numeric(sub.obs['g2m_score'], errors='coerce').fillna(0), max)
print('gradient along pseudotime (Spearman rho):')
for s_ in ['Gast_sm','pSom_sm','Crest_sm','cyc_score']:
    rho,p = spearmanr(t, sub.obs[s_]); print(f'  {s_}: rho={rho:+.3f} p={p:.1e}')

tip 0: Gast=-0.0053 pSom=+0.0498 Crest=+0.0128
tip 2: Gast=-0.0398 pSom=+0.0335 Crest=+0.0303
derived root = tip 0 (Gast/pSom-high, Crest-low)
gradient along pseudotime (Spearman rho):
  Gast_sm: rho=-0.508 p=0.0e+00
  pSom_sm: rho=-0.436 p=0.0e+00
  Crest_sm: rho=+0.370 p=0.0e+00
  cyc_score: rho=+0.168 p=9.6e-68


### House orientation (re-used from 01, verified against pseudotime)
The house orientation — a single rigid rotation of the diagonal developmental axis to
horizontal — was derived **once in 01** (from the manifold's axis of maximal variance,
with the Gast/pSom-high developmental origin placed on the left) and stored as
`X_umap_lin`. Every UMAP deliverable across 01–04 re-uses these exact coordinates, so
orientation is consistent by construction (binding). Here we simply **verify** that layout
now that pseudotime exists: the correlation of the stored `X_umap_lin` x-axis with
pseudotime must be strongly positive (low pseudotime = developmental origin on the left).

In [7]:
t = sub.obs['t'].values
r = float(np.corrcoef(sub.obsm['X_umap_lin'][:,0], t)[0,1])   # re-used from 01, not re-derived
c = sub.obsm['X_umap'] - sub.obsm['X_umap'].mean(0)
dirv = np.array([np.cov(c[:,0], t)[0,1], np.cov(c[:,1], t)[0,1]])
_, _, vt = np.linalg.svd(c, full_matrices=False)
dang = abs(np.degrees(np.arctan2(dirv[1],dirv[0]) - np.arctan2(vt[0,1],vt[0,0]))) % 180
dang = min(dang, 180-dang)
assert r > 0.5, f'stored orientation not pseudotime-consistent (corr={r:.2f})'
print(f'house orientation re-used from 01 (X_umap_lin); corr(x_lin, pseudotime) = {r:+.2f}')
print(f'variance-axis vs pseudotime-axis disagreement = {dang:.1f}deg (small -> equivalent orientation)')

house orientation re-used from 01 (X_umap_lin); corr(x_lin, pseudotime) = +0.96
variance-axis vs pseudotime-axis disagreement = 6.4deg (small -> equivalent orientation)


### Save manifold + trajectory figure (deliverable)

In [8]:
sub.uns['epg'] = {'NodePositions': np.asarray(sub.uns['epg']['NodePositions'])}
sub.write(os.path.join(pth_out, 'manifold.h5ad'))
xl = sub.obsm['X_umap_lin']
xlo,xhi = np.percentile(xl[:,0],[0.3,99.7]); ylo,yhi = np.percentile(xl[:,1],[0.3,99.7])
px=(xhi-xlo)*.05; py=(yhi-ylo)*.08
plot = sub.copy(); plot.obsm['X_umap'] = plot.obsm['X_umap_lin']
fig, ax = plt.subplots(figsize=(3.5,1.5))
sc.pl.umap(plot, color='t', cmap='cividis', size=30, alpha=.8, colorbar_loc='right',
           frameon=False, show=False, ax=ax, title='Pseudotime')
ax.set_xlim(xlo-px,xhi+px); ax.set_ylim(ylo-py,yhi+py)
ax.text(.5,-.12,'Ji et al., 2020', ha='center', transform=ax.transAxes, size=6)
fig.savefig(os.path.join(pth_fig,'manifold.pdf'), bbox_inches='tight')
print('saved manifold.h5ad + manifold.pdf')

saved manifold.h5ad + manifold.pdf
